# Depth Inversion (Method 2 · I2EM physical LUT) → Google Earth (KML/KMZ), streamed via the ESA MAAP STAC API



In [4]:
import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image
from scipy.ndimage import uniform_filter, zoom
from pathlib import Path
import rasterio as rio
import rioxarray as riox


from pystac_client import Client


from DepthInversionBiomass_optionB_copol_lut import (
    mv_from_eps_lut,
    load_i2em_lut_npz,
    retrieve_eps_ks_mv_and_depth_from_i2em_lut, _resample_to_shape, lee_filter,
)

## Credentials & token refresh



In [6]:
CREDENTIALS_FILE = Path(r"C:\Users\Femke.Graveland\OneDrive - ESA\Documents\Python\credentials.txt")

MAAP_IAM_TOKEN_URL = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"


def load_credentials(file_path):
    """Read key=value pairs from a credentials file into a dict."""
    creds = {}
    with open(Path(file_path), "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            creds[key.strip()] = value.strip()
    return creds


def get_token(credential_file_path):
    """Exchange offline token for a short-lived MAAP access token."""
    creds = load_credentials(credential_file_path)
    offline_token = creds.get("OFFLINE_TOKEN")
    client_id = creds.get("CLIENT_ID")
    client_secret = creds.get("CLIENT_SECRET")
    if not all([offline_token, client_id, client_secret]):
        raise ValueError("Missing OFFLINE_TOKEN, CLIENT_ID, or CLIENT_SECRET in credentials file")
    response = requests.post(MAAP_IAM_TOKEN_URL, data={
        "client_id": client_id, "client_secret": client_secret,
        "grant_type": "refresh_token", "refresh_token": offline_token,
        "scope": "offline_access openid",
    })
    response.raise_for_status()

    access_token = response.json().get("access_token")
    if not access_token:
        raise RuntimeError("Failed to retrieve access token")
    return access_token

## Connect to the catalog

In [7]:
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from pystac_client.stac_api_io import StacApiIO

MAAP_CATALOG_URL = "https://catalog.maap.eo.esa.int/catalogue/"


_retry = Retry(
    total=6, connect=6, read=6, status=6,
    backoff_factor=1.5,                       # waits ~0, 1.5, 3, 6, 12, 24 s
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=True,
)
_stac_io = StacApiIO()
_stac_io.session.mount("https://", HTTPAdapter(max_retries=_retry))
_stac_io.session.mount("http://", HTTPAdapter(max_retries=_retry))

catalog = Client.open(MAAP_CATALOG_URL, stac_io=_stac_io)
print("Connected to MAAP STAC catalogue (HTTP retry/backoff on 5xx enabled)")

Connected to MAAP STAC catalogue (HTTP retry/backoff on 5xx enabled)


## Configuration



In [31]:
# rch 
AOI_NAME = "Nambia"
COLLECTIONS = ["BiomassLevel1c"]

BBOX = [15, -27, 16, -23] #Nambia
# BBOX = [21.783476, 8.645889, 26.349895, 29.028335]  # [west, south, east, north] Sahara
DATETIME_RANGE = ["2026-01-01T00:00:00Z", "2026-12-31T00:00:00Z"]
MAX_ITEMS = 700

TRACK_FILTER = ["003","040","018"]   # list of strings, or None to process everything
FRAME_FILTER = "290"      # e.g. ["131", "132"]

#  Output 
OUTPUT_DIR = Path(r"C:\Users\Femke.Graveland\OneDrive - ESA\Documents\DATA\1C Sahara\Google Earth Output (MAAP DI Method2_v2)")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_REFRESH_EVERY = 10   # refresh token every N frame-pairs processed

# hod 2 depth inversion (I2EM physical LUT) 
I2EM_LUT_PATH = "biomass_i2em_main_v9.npz"   # EDIT: path to your generated I2EM LUT
DIELECTRIC_MODEL = "topp"   # "dobson" | "crim" | "topp"  -> moisture model on retrieved eps'

# Lee speckle filter on sigma0 BEFORE inversion (applied to both co-pols),
# matching Model 2 in the comparison notebook.
USE_LEE_FILTER = True
LEE_SIZE = 5


ML_AZ = 12   # azimuth (row) looks
ML_RG = 2    # range (column) looks

import numpy as np
import warnings as _warnings

def _multilook(a, fy, fx, valid_max=None):
    """Incoherent multilook by block-averaging fy x fx pixels (nan-aware).
    Masks non-finite (and out-of-range fill, if valid_max given) before the mean
    so margin fill values don't bleed into interior blocks. Trailing rows/cols
    that don't fill a whole block are dropped."""
    a = np.asarray(a, dtype=float)
    if fy == 1 and fx == 1:
        return a
    ny, nx = a.shape
    ny2, nx2 = (ny // fy) * fy, (nx // fx) * fx
    a = a[:ny2, :nx2]
    bad = ~np.isfinite(a)
    if valid_max is not None:
        bad = bad | (a <= 0) | (a >= valid_max)
    a = np.where(bad, np.nan, a)
    blk = a.reshape(ny2 // fy, fy, nx2 // fx, fx)
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(blk, axis=(1, 3))


I2EM_KWARGS = dict(
    channels=("hh", "vv"),
    sigma0_input_units="db",     # Method 2 expects dB input
    eps_min=None, eps_max=None,  # unconstrained eps' (same as Model 2 default)
    ks_min=None, ks_max=0.45,    # same ks upper bound as Model 2
    ks_prior=None,               # honest, unconstrained ks (USE_KS_PRIOR=False in M2)
    sigma_ks_prior=None,
    theta_bin_step=0.25,
    dielectric_model=DIELECTRIC_MODEL,
    use_volume_kz=True,
    depth_model="exponential",
    gamma_min=0.1,
    sand_pct=85.0, clay_pct=5.0, pb=1.3,
)

LAYER_SPECS = [
    ("VV backscatter (dB)",   "vv_db", "gray",    -30,  0),
    ("Volumetric moisture",   "mv",    "inferno", None, None),
    ("Dielectric constant",   "eps",   "viridis", None, None),
    ("Penetration depth (m)", "depth", "magma",    0,   10),
]


I2EM_TILE_ROWS = 512   # rows per tile; lower this if you hit MemoryError early
MV_TILE_ROWS = 256     # rows per tile for the eps'->mv nearest-grid lookup
                       # (mv_from_eps_lut broadcasts an (n_mv, ny, nx) array;
                       #  full-res would need tens of GiB, so tile it too)
MIN_TILE_ROWS = 50     # floor tile size


PREVIEW_DECIMATE = 4   # 1 = full resolution; 4 = every 4th pixel (~16x fewer pixels)

BATCH_DECIMATE = 1

MAX_PAIRS = 1


SKIP_EXISTING = True


PNG_MAX_DIM = 2400      # overlays are downsampled to this before rendering
PCTL_SUBSAMPLE = 2_000_000   # max pixels used to compute 2/98 percentiles

# BIOMASS nominal: slant range ~800 km; perpendicular baseline varies 100-800 m.
FALLBACK_B_PERP_M = 300.0
FALLBACK_SLANT_RANGE_M = 800_000.0


In [9]:
# d the I2EM physical LUT (Method 2 forward model) 
# The npz stores LINEAR power (not dB) — see the prior units-mismatch fix that
# was pinning every pixel at a LUT boundary corner.
i2em_lut = load_i2em_lut_npz(I2EM_LUT_PATH, sigma0_units="linear")
print("Loaded I2EM LUT:", I2EM_LUT_PATH)
print("  LUT theta axis:", i2em_lut.theta_axis)


Loaded I2EM LUT: biomass_i2em_main_v9.npz
  LUT theta axis: [22.         22.10077519 22.20155039 22.30232558 22.40310078 22.50387597
 22.60465116 22.70542636 22.80620155 22.90697674 23.00775194 23.10852713
 23.20930233 23.31007752 23.41085271 23.51162791 23.6124031  23.71317829
 23.81395349 23.91472868 24.01550388 24.11627907 24.21705426 24.31782946
 24.41860465 24.51937984 24.62015504 24.72093023 24.82170543 24.92248062
 25.02325581 25.12403101 25.2248062  25.3255814  25.42635659 25.52713178
 25.62790698 25.72868217 25.82945736 25.93023256 26.03100775 26.13178295
 26.23255814 26.33333333 26.43410853 26.53488372 26.63565891 26.73643411
 26.8372093  26.9379845  27.03875969 27.13953488 27.24031008 27.34108527
 27.44186047 27.54263566 27.64341085 27.74418605 27.84496124 27.94573643
 28.04651163 28.14728682 28.24806202 28.34883721 28.4496124  28.5503876
 28.65116279 28.75193798 28.85271318 28.95348837 29.05426357 29.15503876
 29.25581395 29.35658915 29.45736434 29.55813953 29.65891473 29.7

In [10]:
def clean_sigma0_linear_to_db(sigma0_linear):
    """Linear sigma0 -> dB, masking fill / non-physical values.

    Ported from the Model 2 cell: removes the ~9.998e7 fill values and any
    non-positive / non-finite entries before the log.
    """
    sigma0_linear = np.asarray(sigma0_linear, dtype=float)
    valid = (
        np.isfinite(sigma0_linear)
        & (sigma0_linear > 0)
        & (sigma0_linear < 10)   # removes 9.998e7 fill values
    )
    sigma0_db = np.full_like(sigma0_linear, np.nan, dtype=float)
    sigma0_db[valid] = 10.0 * np.log10(sigma0_linear[valid])
    return sigma0_db, valid


## Search + group by (track, frame), take C01 → C02 pairs



In [18]:
import xml.etree.ElementTree as ET
import xarray as xr


# 
# Streaming annotation XML and LUT NetCDF from STAC assets
# 

def _stream_annotation_xml(item, token):
    """
    Download annotation_coregistered XML and return the parsed root.
    The MAAP catalog exposes two annotation XMLs as enclosure_annot_xml_1
    and enclosure_annot_xml_2. One is the coregistered annotation (has
    staCoregistrationParameters/normalBaseline), the other is primary.
    We try both and return the one that contains the baseline.
    """
    annot_keys = [k for k in item.assets if "annot_xml" in k]
    if not annot_keys:
        return None

    for key in sorted(annot_keys):
        href = item.assets[key].href
        try:
            resp = requests.get(href, headers={"Authorization": f"Bearer {token}"})
            resp.raise_for_status()
            root = ET.fromstring(resp.content)
            if root.find(".//staCoregistrationParameters") is not None or \
               root.find(".//normalBaseline") is not None:
                return root
        except Exception:
            continue

    # None had the baseline tag — return the first one parsed anyway
    href = item.assets[annot_keys[0]].href
    resp = requests.get(href, headers={"Authorization": f"Bearer {token}"})
    resp.raise_for_status()
    return ET.fromstring(resp.content)


def _stream_lut_nc_to_tempfile(item, token):
    """Download the LUT NetCDF (enclosure_nc) to a temp file."""
    key = "enclosure_nc"
    if key not in item.assets:
        for k in item.assets:
            if k.endswith("_nc") or "_lut" in k:
                key = k
                break
        else:
            return None

    href = item.assets[key].href
    resp = requests.get(href, headers={"Authorization": f"Bearer {token}"},
                        stream=True)
    resp.raise_for_status()
    tmp = tempfile.NamedTemporaryFile(suffix=".nc", delete=False)
    for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
        tmp.write(chunk)
    tmp.close()
    return tmp.name


def _parse_baseline_from_xml(xml_root):
    """Extract normalBaseline from annotation_coregistered XML."""
    for xpath in ("./staCoregistrationParameters/normalBaseline",
                  ".//normalBaseline",
                  ".//perpendicularBaseline",
                  ".//Bperp"):
        el = xml_root.find(xpath)
        if el is not None and el.text:
            try:
                return float(el.text.strip())
            except ValueError:
                continue
    return None


def _parse_slant_range_from_nc(nc_path):
    """Extract slant range from the LUT NetCDF root variables."""
    c0 = 299792458.0
    try:
        ds = xr.open_dataset(nc_path)
        for var_name in ("slantRangeTimeSLC", "slantRangeTimeRGC",
                         "slantRangeTimeRAW", "slantRangeTime"):
            if var_name in ds.variables:
                t = float(np.nanmedian(ds[var_name].values))
                if np.isfinite(t) and t > 0:
                    ds.close()
                    return 0.5 * c0 * t
        for var_name in ("slantRange", "slantRangeDistance",
                         "rangeDistance", "slantRangeMean"):
            if var_name in ds.variables:
                r = float(np.nanmedian(ds[var_name].values))
                if np.isfinite(r) and r > 0:
                    ds.close()
                    return r
        ds.close()
    except Exception as e:
        print(f"    WARNING: failed to read LUT NetCDF: {e}")
    return None


def _parse_incidence_from_nc(nc_path, target_shape):
    """Try to read incidence angle from the LUT NetCDF geometry group."""
    try:
        ds = xr.open_dataset(nc_path, group="geometry")
        if "incidenceAngle" in ds.variables:
            inc = ds["incidenceAngle"].values
            ds.close()
            if inc.ndim == 2:
                return _resample_to_shape(inc, target_shape)
            else:
                return np.full(target_shape, float(np.nanmedian(inc)))
        ds.close()
    except Exception:
        pass
    return None


def get_baseline_and_slant(item, token):
    """
    Extract perpendicular baseline and slant range by streaming the
    product's annotation XML and LUT NetCDF from their STAC assets.
    Falls back to configured constants if the files can't be read.
    """
    b_perp = None
    r_slant = None

    # ── 1. Annotation XML → baseline ─────────────────────────────────────────
    try:
        xml_root = _stream_annotation_xml(item, token)
        if xml_root is not None:
            b_perp = _parse_baseline_from_xml(xml_root)
            if b_perp is not None:
                print(f"    Baseline from annotation XML: {b_perp:.2f} m")
            else:
                print(f"    WARNING: annotation XML streamed but normalBaseline tag not found")
        else:
            print(f"    WARNING: no annotation XML asset found (expected enclosure_annot_xml_*)")
    except Exception as e:
        print(f"    WARNING: could not stream annotation XML: {e}")

    # ── 2. LUT NetCDF → slant range ──────────────────────────────────────────
    nc_path = None
    try:
        nc_path = _stream_lut_nc_to_tempfile(item, token)
        if nc_path is not None:
            r_slant = _parse_slant_range_from_nc(nc_path)
            if r_slant is not None:
                print(f"    Slant range from LUT NetCDF: {r_slant:.2f} m")
            else:
                print(f"    WARNING: LUT NetCDF streamed but no slant range variable found")
        else:
            print(f"    WARNING: no LUT NetCDF asset found (expected enclosure_nc)")
    except Exception as e:
        print(f"    WARNING: could not stream LUT NetCDF: {e}")
    finally:
        if nc_path is not None:
            try:
                Path(nc_path).unlink()
            except OSError:
                pass

    # ── 3. Fallback constants ────────────────────────────────────────────────
    if b_perp is None:
        print(f"    WARNING: b_perp not found — using fallback {FALLBACK_B_PERP_M} m")
        b_perp = FALLBACK_B_PERP_M
    if r_slant is None:
        print(f"    WARNING: slant_range not found — using fallback {FALLBACK_SLANT_RANGE_M} m")
        r_slant = FALLBACK_SLANT_RANGE_M

    return float(b_perp), float(r_slant)


def get_incidence_angle_from_nc(item, token, target_shape):
    """
    Stream the LUT NetCDF and read incidence angle from the geometry group.
    Falls back to a linear ramp if not available.
    """
    ny, nx = target_shape
    nc_path = None
    try:
        nc_path = _stream_lut_nc_to_tempfile(item, token)
        if nc_path is not None:
            inc = _parse_incidence_from_nc(nc_path, target_shape)
            if inc is not None:
                print(f"    Incidence angle from LUT NetCDF geometry group")
                return inc
    except Exception as e:
        print(f"    WARNING: could not read incidence from LUT: {e}")
    finally:
        if nc_path is not None:
            try:
                Path(nc_path).unlink()
            except OSError:
                pass

    print(f"    WARNING: using 23-38 deg ramp for incidence angle")
    ramp = np.linspace(23.0, 38.0, nx)
    return np.broadcast_to(ramp, (ny, nx)).copy()


def get_item_quad_coordinates(item):
    """Footprint as a KML gx:LatLonQuad coordinate string, from the STAC geometry."""
    geom = item.geometry
    if geom is None:
        raise ValueError(f"No geometry on item {item.id}")
    if geom["type"] == "Polygon":
        coords = geom["coordinates"][0]
    elif geom["type"] == "MultiPolygon":
        coords = geom["coordinates"][0][0]
    else:
        raise ValueError(f"Unsupported geometry type: {geom['type']}")
    if coords[0] == coords[-1]:
        coords = coords[:-1]
    if len(coords) < 4:
        raise ValueError(f"Not enough footprint corners for {item.id}")
    return " ".join(f"{lon},{lat},0" for lon, lat in coords[:4])

In [19]:
import re
import time
import xml.etree.ElementTree as ET
import numpy as np

from collections import defaultdict, Counter



# Regex patterns


PRODUCT_RE = re.compile(r"(S[123]_STA__1S)")
SCENE_RE = re.compile(r"_T(\d{3})_F(\d{3})_")
CYCLE_RE = re.compile(r"_C(\d{2})_")
MAJOR_CYCLE_RE = re.compile(r"_M(\d{2})_")
ACQ_TIME_RE = re.compile(r"_(\d{8}T\d{6})_")



# Preferred baseline-pair selection settings
PREFERRED_MASTER_CYCLES = {"C04"}

# Ordered preference: C03 first, C05 second
PREFERRED_SECONDARY_CYCLE_ORDER = ["C03", "C05"]

TARGET_ABS_BPERP_M = 2500.0
PREFERRED_ABS_BPERP_RANGE_M = (1500.0, 2500.0)



# Basic metadata helpers


def track_frame(item_id):
    """
    Extract BIOMASS track and frame from product ID.
    """
    m = SCENE_RE.search(item_id)
    return (m.group(1), m.group(2)) if m else (None, None)


def cycle_label(item):
    """
    Return cycle label, e.g. C01, C02.
    """
    m = CYCLE_RE.search(item.id)
    return f"C{m.group(1)}" if m else "C??"


def cycle_number(item):
    """
    Return cycle number as integer.
    """
    m = CYCLE_RE.search(item.id)
    return int(m.group(1)) if m else None


def cycle_number_from_label(cyc):
    """
    Convert cycle label like C04 to integer 4.
    """
    m = re.search(r"C(\d{2})", str(cyc))
    return int(m.group(1)) if m else None


def major_cycle(item):
    """
    Return major cycle label, e.g. M02, M03.
    """
    m = MAJOR_CYCLE_RE.search(item.id)
    return f"M{m.group(1)}" if m else "M??"


def acquisition_time(item):
    """
    Extract acquisition start time from BIOMASS product ID.
    """
    m = ACQ_TIME_RE.search(item.id)
    return m.group(1) if m else ""


def product_type(item):
    """
    Get product type either from STAC properties or from product ID.
    """
    pt = item.properties.get("productType")

    if pt in {"S1_STA__1S", "S2_STA__1S", "S3_STA__1S"}:
        return pt

    m = PRODUCT_RE.search(item.id)
    return m.group(1) if m else None


def orbit_direction(item):
    """
    Return orbit direction.

    The catalogue may store this under different property names.
    """
    candidates = [
        "orbitDirection",
        "orbit_direction",
        "sat:orbit_state",
        "sat:orbitState",
    ]

    for key in candidates:
        value = item.properties.get(key)
        if value is not None:
            return str(value).upper()

    return "UNKNOWN"



# Duplicate-cycle splitting helpers


def duplicate_cycle_index(item, scenes_in_base_group):
    """
    Return duplicate index D01, D02, ... for products with duplicate cycles.

    The base group already includes major cycle, so D01/D02 only separates
    remaining duplicate cycles within the same major cycle.
    """

    cyc = cycle_label(item)

    same_cycle_items = [
        s for s in scenes_in_base_group
        if cycle_label(s) == cyc
    ]

    same_cycle_items = sorted(
        same_cycle_items,
        key=lambda s: (acquisition_time(s), s.id),
    )

    if len(same_cycle_items) == 1:
        return "D01"

    idx = same_cycle_items.index(item) + 1
    return f"D{idx:02d}"


def sort_scene_key(item):
    """
    Sorting key for scenes inside each group.
    """
    cyc_num = cycle_number(item)
    if cyc_num is None:
        cyc_num = 999

    return (
        cyc_num,
        acquisition_time(item),
        item.id,
    )



# XML / annotation helpers


def _strip_ns(tag):
    """
    Remove XML namespace from tag name.
    """
    return tag.split("}", 1)[-1] if "}" in tag else tag


def _scene_signature_from_id(scene_id):
    """
    Extract acquisition time, cycle, major cycle, track, and frame from a BIOMASS ID.

    Works for both STA and SCS product names, so it can match e.g.
    BIO_S2_STA__... from STAC to BIO_S2_SCS__... in annotation XML.
    """

    time_match = re.search(r"_(\d{8}T\d{6})_", scene_id)
    cycle_match = re.search(r"_C(\d{2})_", scene_id)
    major_cycle_match = re.search(r"_M(\d{2})_", scene_id)
    track_frame_match = re.search(r"_T(\d{3})_F(\d{3})_", scene_id)

    if not (time_match and cycle_match and track_frame_match):
        return None

    return {
        "time": time_match.group(1),
        "cycle": cycle_match.group(1),
        "major_cycle": major_cycle_match.group(1) if major_cycle_match else None,
        "track": track_frame_match.group(1),
        "frame": track_frame_match.group(2),
    }


def _same_scene_identity(id_a, id_b):
    """
    Compare two BIOMASS product IDs by acquisition time, cycle, major cycle,
    
    track, and frame.

    This allows matching STA STAC item IDs to SCS annotation IDs.

    If one ID does not contain major cycle, comparison falls back to:
        time, cycle, track, frame
    """

    sig_a = _scene_signature_from_id(id_a)
    sig_b = _scene_signature_from_id(id_b)

    if sig_a is None or sig_b is None:
        return False

    required_fields = ["time", "cycle", "track", "frame"]

    for field in required_fields:
        if sig_a[field] != sig_b[field]:
            return False

    if sig_a["major_cycle"] is not None and sig_b["major_cycle"] is not None:
        if sig_a["major_cycle"] != sig_b["major_cycle"]:
            return False

    return True


def _parse_coreg_baseline_entries(xml_root):
    """
    Parse all staCoregistrationParameters entries containing:
        primaryImage
        secondaryImage
        normalBaseline

    Returns a list of dictionaries.
    """

    entries = []

    for elem in xml_root.iter():

        if _strip_ns(elem.tag) != "staCoregistrationParameters":
            continue

        primary = None
        secondary = None
        baseline = None

        for child in elem:
            tag = _strip_ns(child.tag)
            text = (child.text or "").strip()

            if tag == "primaryImage":
                primary = text

            elif tag == "secondaryImage":
                secondary = text

            elif tag == "normalBaseline":
                try:
                    baseline = float(text)
                except Exception:
                    baseline = None

        if primary and secondary and baseline is not None:
            entries.append(
                {
                    "primaryImage": primary,
                    "secondaryImage": secondary,
                    "normalBaseline": baseline,
                }
            )

    return entries



# Preferred candidate scoring


def _candidate_score_for_preferred_pair(candidate):
    """
    Score candidate baseline pair.

    Lower score is better.

    Priority:
      1. Prefer master/primary cycle C04.
      2. Prefer secondary C03 first, then C05.
      3. Prefer |B_perp| in the range 500--1100 m.
      4. Prefer |B_perp| closest to TARGET_ABS_BPERP_M.
    """

    p_cyc = candidate["primary_cycle"]
    s_cyc = candidate["secondary_cycle"]
    b_abs = abs(candidate["b_perp_m"])

    # Primary/master should be C04.
    if p_cyc in PREFERRED_MASTER_CYCLES:
        master_penalty = 0
    else:
        master_penalty = 10000

    # Strong ordered preference for secondary cycle.
    if s_cyc in PREFERRED_SECONDARY_CYCLE_ORDER:
        secondary_rank = PREFERRED_SECONDARY_CYCLE_ORDER.index(s_cyc)
        secondary_penalty = secondary_rank * 10
    else:
        secondary_penalty = 1000

    # Penalise candidates outside preferred abs baseline range.
    range_penalty = 0
    if PREFERRED_ABS_BPERP_RANGE_M is not None:
        bmin, bmax = PREFERRED_ABS_BPERP_RANGE_M
        if not (bmin <= b_abs <= bmax):
            range_penalty = 500

    baseline_distance = abs(b_abs - TARGET_ABS_BPERP_M)

    return (
        master_penalty
        + secondary_penalty
        + range_penalty
        + baseline_distance
    )


def select_pair_from_annotation_baseline(scenes, token, verbose=True):
    """
    Select the preferred master-secondary scene pair from Level-1C coregistration
    annotation.

    This version does NOT return the first valid annotation entry.

    It collects all valid entries and selects the preferred intermediate-baseline
    pair:

        C04 -> C03, preferred first
        C04 -> C05, preferred second

    These are expected to correspond approximately to:
        B_perp around -800 m or +800 m.

    Bad entries are rejected:
      - self-pair, e.g. C04 -> C04
      - same major cycle + same cycle
      - invalid or near-zero B_perp

    Returns
    
    master_item : STAC item
    secondary_item : STAC item
    b_perp_m : float
    b_perp_source : str
    """

    rejected_entries = []
    valid_candidates = []

    for annotation_source_item in scenes:

        try:
            xml_root = _stream_annotation_xml(annotation_source_item, token)

        except Exception as e:
            if verbose:
                print(
                    f"    WARNING: could not stream annotation XML from "
                    f"{cycle_label(annotation_source_item)} "
                    f"{major_cycle(annotation_source_item)}: {e}"
                )
            continue

        if xml_root is None:
            continue

        entries = _parse_coreg_baseline_entries(xml_root)

        if not entries:
            continue

        if verbose:
            print(
                f"    Found {len(entries)} coregistration baseline entrie(s) "
                f"in annotation from {cycle_label(annotation_source_item)} "
                f"{major_cycle(annotation_source_item)}"
            )

        for entry in entries:

            primary_id = entry["primaryImage"]
            secondary_id = entry["secondaryImage"]
            b_perp = entry["normalBaseline"]

            primary_item = None
            secondary_item = None

            for s in scenes:

                if _same_scene_identity(s.id, primary_id):
                    primary_item = s

                if _same_scene_identity(s.id, secondary_id):
                    secondary_item = s

            if primary_item is None or secondary_item is None:
                if verbose:
                    print("    Baseline entry did not match available scenes:")
                    print(f"      primaryImage   = {primary_id}")
                    print(f"      secondaryImage = {secondary_id}")
                    print(f"      normalBaseline = {b_perp:+.2f} m")
                continue

            p_mc = major_cycle(primary_item)
            s_mc = major_cycle(secondary_item)
            p_cyc = cycle_label(primary_item)
            s_cyc = cycle_label(secondary_item)

            # Reject self-pair
            if primary_item.id == secondary_item.id:
                msg = (
                    f"self-pair rejected: "
                    f"{p_mc} {p_cyc} -> {s_mc} {s_cyc} | "
                    f"B_perp={b_perp:+.6f} m"
                )
                rejected_entries.append(msg)

                if verbose:
                    print(f"    Rejected {msg}")

                continue

            # Reject same-cycle pair
            if p_mc == s_mc and p_cyc == s_cyc:
                msg = (
                    f"same-cycle pair rejected: "
                    f"{p_mc} {p_cyc} -> {s_mc} {s_cyc} | "
                    f"B_perp={b_perp:+.6f} m"
                )
                rejected_entries.append(msg)

                if verbose:
                    print(f"    Rejected {msg}")

                continue

            # Reject invalid/near-zero baseline
            if b_perp is None or not np.isfinite(b_perp) or abs(float(b_perp)) < 1e-3:
                msg = (
                    f"invalid/near-zero baseline rejected: "
                    f"{p_mc} {p_cyc} -> {s_mc} {s_cyc} | "
                    f"B_perp={b_perp:+.6f} m"
                )
                rejected_entries.append(msg)

                if verbose:
                    print(f"    Rejected {msg}")

                continue

            candidate = {
                "primary_item": primary_item,
                "secondary_item": secondary_item,
                "b_perp_m": float(b_perp),
                "annotation_source_item": annotation_source_item,
                "primary_major_cycle": p_mc,
                "secondary_major_cycle": s_mc,
                "primary_cycle": p_cyc,
                "secondary_cycle": s_cyc,
                "primary_id": primary_item.id,
                "secondary_id": secondary_item.id,
            }

            valid_candidates.append(candidate)

            if verbose:
                print(
                    f"    Valid candidate: "
                    f"{p_mc} {p_cyc} -> {s_mc} {s_cyc} | "
                    f"B_perp={b_perp:+.2f} m"
                )

    if not valid_candidates:
        msg = "No valid annotation normalBaseline entry could be matched to available scenes."

        if rejected_entries:
            msg += " Rejected entries included: " + " | ".join(rejected_entries[:5])

        raise RuntimeError(msg)

    # Rank all candidates and select the best one.
    valid_candidates = sorted(
        valid_candidates,
        key=_candidate_score_for_preferred_pair,
    )

    selected = valid_candidates[0]

    master_item = selected["primary_item"]
    secondary_item = selected["secondary_item"]
    b_perp_m = selected["b_perp_m"]

    if verbose:
        print("\n    Candidate ranking:")
        for cand in valid_candidates:
            score = _candidate_score_for_preferred_pair(cand)
            print(
                f"      score={score:8.2f} | "
                f"{cand['primary_major_cycle']} {cand['primary_cycle']} -> "
                f"{cand['secondary_major_cycle']} {cand['secondary_cycle']} | "
                f"B_perp={cand['b_perp_m']:+.2f} m"
            )

        print(
            f"\n    SELECTED preferred intermediate-baseline pair: "
            f"{major_cycle(master_item)} {cycle_label(master_item)} -> "
            f"{major_cycle(secondary_item)} {cycle_label(secondary_item)} | "
            f"B_perp={b_perp_m:+.2f} m"
        )

    return (
        master_item,
        secondary_item,
        b_perp_m,
        "normalBaseline_preferred_C04_to_C03_or_C05_intermediate_baseline",
    )



# Search helper


PAGE_LIMIT = 50


def collect_items(search, attempts=4, pause=8):
    """
    Collect all matching STAC items, retrying on transient API errors.
    """
    from pystac_client.exceptions import APIError

    last = None

    for k in range(1, attempts + 1):

        try:
            return list(search.items())

        except APIError as e:
            last = e
            first = str(e).strip().splitlines()[0][:120]

            print(
                f"  search attempt {k}/{attempts} failed "
                f"({first}) - retrying in {pause}s..."
            )

            time.sleep(pause)
            pause = int(pause * 1.6)

    print(
        "  all retries exhausted - the MAAP gateway appears to be down "
        "right now; try again later."
    )

    raise last



# Ensure token exists for annotation streaming


try:
    token
except NameError:
    token = get_token(CREDENTIALS_FILE)



# Search catalogue


PRODUCT_FILTER = (
    "productType='S2_STA__1S' OR "
    "productType='S1_STA__1S' OR "
    "productType='S3_STA__1S'"
)

search = catalog.search(
    method="GET",
    limit=PAGE_LIMIT,
    bbox=BBOX,
    datetime=DATETIME_RANGE,
    max_items=MAX_ITEMS,
    collections=COLLECTIONS,
    filter=PRODUCT_FILTER,
)

items = collect_items(search)

try:
    _matched = search.matched()
except Exception:
    _matched = "unknown"

print(
    f"Found {len(items)} item(s) "
    f"(matched: {_matched}, cap: {MAX_ITEMS}, page size: {PAGE_LIMIT})"
)



# Parse basic metadata


parsed = []

for item in items:

    pt = product_type(item)
    orbit = orbit_direction(item)
    t, f = track_frame(item.id)
    mc = major_cycle(item)

    parsed.append(
        (
            item,
            pt,
            orbit,
            t,
            f,
            mc,
        )
    )



# First-level grouping:
# product type, orbit direction, track, frame, major cycle


base_groups = defaultdict(list)

for item, pt, orbit, t, f, mc in parsed:

    if pt is None:
        continue

    if orbit is None:
        continue

    if t is None or f is None:
        continue

    if mc is None:
        continue

    base_groups[(pt, orbit, t, f, mc)].append(item)

base_groups = {
    k: sorted(v, key=sort_scene_key)
    for k, v in sorted(base_groups.items())
}



# Print available combinations before duplicate-cycle splitting


print("\nAvailable before duplicate-cycle splitting:")

for (pt, orbit, t, f, mc), scenes in base_groups.items():

    cycles = [cycle_label(s) for s in scenes]
    cycle_counts = Counter(cycles)

    duplicate_cycles = {
        c: n for c, n in cycle_counts.items()
        if n > 1
    }

    print(
        f"  {pt} {orbit} T{t}_F{f} {mc} "
        f"({len(scenes)} scene(s), cycles={cycles})"
    )

    if duplicate_cycles:
        print(f"    duplicate cycles within {mc}: {duplicate_cycles}")



# Split duplicate-cycle groups inside each major cycle


split_groups = defaultdict(list)

for (pt, orbit, t, f, mc), scenes in base_groups.items():

    for item in scenes:

        dup_idx = duplicate_cycle_index(item, scenes)

        split_groups[(pt, orbit, t, f, mc, dup_idx)].append(item)

split_groups = {
    k: sorted(v, key=sort_scene_key)
    for k, v in sorted(split_groups.items())
}



# Print available combinations after splitting


print("\nAvailable after duplicate-cycle splitting:")

for (pt, orbit, t, f, mc, dup_idx), scenes in split_groups.items():

    cycles = [cycle_label(s) for s in scenes]

    print(
        f"  {pt} {orbit} T{t}_F{f} {mc} {dup_idx} "
        f"({len(scenes)} scene(s), cycles={cycles})"
    )



# Diagnostic printout for duplicate stacks


PRINT_DUPLICATE_DIAGNOSTICS = True

if PRINT_DUPLICATE_DIAGNOSTICS:

    print("\nDuplicate-stack diagnostics:")

    for key, scenes in split_groups.items():

        pt, orbit, t, f, mc, dup_idx = key

        base_key = (pt, orbit, t, f, mc)
        base_cycles = [cycle_label(s) for s in base_groups[base_key]]
        base_counts = Counter(base_cycles)

        has_duplicates = any(n > 1 for n in base_counts.values())

        if not has_duplicates:
            continue

        print("\n" + "-" * 80)
        print(f"{pt} {orbit} T{t}_F{f} {mc} {dup_idx}")

        for s in scenes:
            print(
                f"  {major_cycle(s)}  "
                f"{cycle_label(s)}  "
                f"{acquisition_time(s)}  "
                f"{s.id}"
            )



# Optional filters


try:
    TRACK_FILTER
except NameError:
    TRACK_FILTER = None

try:
    FRAME_FILTER
except NameError:
    FRAME_FILTER = None

try:
    ORBIT_FILTER
except NameError:
    ORBIT_FILTER = None

try:
    MAJOR_CYCLE_FILTER
except NameError:
    MAJOR_CYCLE_FILTER = None

try:
    DUPLICATE_FILTER
except NameError:
    DUPLICATE_FILTER = None

try:
    PRODUCT_TYPE_FILTER
except NameError:
    PRODUCT_TYPE_FILTER = None


filtered_groups = {}

for (pt, orbit, t, f, mc, dup_idx), scenes in split_groups.items():

    if PRODUCT_TYPE_FILTER is not None and pt not in PRODUCT_TYPE_FILTER:
        continue

    if TRACK_FILTER is not None and t not in TRACK_FILTER:
        continue

    if FRAME_FILTER is not None and f not in FRAME_FILTER:
        continue

    if ORBIT_FILTER is not None and orbit not in ORBIT_FILTER:
        continue

    if MAJOR_CYCLE_FILTER is not None and mc not in MAJOR_CYCLE_FILTER:
        continue

    if DUPLICATE_FILTER is not None and dup_idx not in DUPLICATE_FILTER:
        continue

    filtered_groups[(pt, orbit, t, f, mc, dup_idx)] = scenes



# Keep only groups with at least two scenes


candidate_groups = {
    k: v for k, v in filtered_groups.items()
    if len(v) >= 2
}

short_groups = {
    k: v for k, v in filtered_groups.items()
    if len(v) < 2
}

print(
    f"\n{len(candidate_groups)} candidate stack group(s) "
    f"with >=2 scenes."
)

if short_groups:
    print(
        f"{len(short_groups)} group(s) skipped "
        f"because only 1 scene was found:"
    )

    for k, v in short_groups.items():
        print(f"  {k}: {len(v)} scene(s)")



# Select preferred intermediate-baseline master-secondary pairs


pairs = {}
baseline_by_key = {}
baseline_match_failures = {}

print("\nSelecting preferred intermediate-baseline master-secondary pairs:")

for (pt, orbit, t, f, mc, dup_idx), scenes in candidate_groups.items():

    key = (pt, orbit, t, f, mc, dup_idx)

    print("\n" + "-" * 80)
    print(f"{pt} {orbit} T{t}_F{f} {mc} {dup_idx}")
    print(
        "Available cycles: "
        f"{[(major_cycle(s), cycle_label(s)) for s in scenes]}"
    )

    try:
        master_item, secondary_item, b_perp_m, b_perp_source = (
            select_pair_from_annotation_baseline(
                scenes=scenes,
                token=token,
                verbose=True,
            )
        )

        pairs[key] = [master_item, secondary_item]

        baseline_by_key[key] = {
            "b_perp_m": b_perp_m,
            "b_perp_source": b_perp_source,
            "baseline_major_cycle": mc,
            "baseline_master": cycle_label(master_item),
            "baseline_secondary": cycle_label(secondary_item),
            "baseline_master_major_cycle": major_cycle(master_item),
            "baseline_secondary_major_cycle": major_cycle(secondary_item),
            "baseline_master_id": master_item.id,
            "baseline_secondary_id": secondary_item.id,
            "baseline_master_time": acquisition_time(master_item),
            "baseline_secondary_time": acquisition_time(secondary_item),
        }

        print(
            f"  SELECTED: "
            f"master={major_cycle(master_item)} {cycle_label(master_item)}  "
            f"secondary={major_cycle(secondary_item)} {cycle_label(secondary_item)}  "
            f"B_perp={b_perp_m:+.2f} m  "
            f"[{b_perp_source}]"
        )

    except Exception as e:

        baseline_match_failures[key] = str(e)

        print(
            f"  SKIPPED: no valid preferred baseline-matched pair found "
            f"({type(e).__name__}: {e})"
        )



# Final summary


print(
    f"\n{len(pairs)} preferred baseline-matched pair(s) ready for depth inversion."
)

if baseline_match_failures:

    print(
        f"\n{len(baseline_match_failures)} candidate group(s) skipped because "
        f"no valid preferred annotation baseline could be matched:"
    )

    for k, err in baseline_match_failures.items():
        print(f"  {k}: {err}")


print("\nFinal selected preferred intermediate-baseline master-secondary pairs:")

for (pt, orbit, t, f, mc, dup_idx), scenes_pair in pairs.items():

    master_item, secondary_item = scenes_pair
    binfo = baseline_by_key[(pt, orbit, t, f, mc, dup_idx)]

    print(
        f"  {pt} {orbit} T{t}_F{f} {mc} {dup_idx}: "
        f"master={major_cycle(master_item)} {cycle_label(master_item)}  "
        f"secondary={major_cycle(secondary_item)} {cycle_label(secondary_item)}  "
        f"B_perp={binfo['b_perp_m']:+.2f} m  "
        f"[{binfo['b_perp_source']}]"
    )

Found 138 item(s) (matched: 138, cap: 700, page size: 50)

Available before duplicate-cycle splitting:
  S1_STA__1S ASCENDING T003_F287 M03 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S1_STA__1S ASCENDING T003_F288 M03 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S1_STA__1S ASCENDING T003_F289 M03 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S1_STA__1S ASCENDING T003_F290 M02 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S1_STA__1S ASCENDING T003_F290 M03 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S1_STA__1S ASCENDING T003_F291 M02 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S1_STA__1S ASCENDING T003_F291 M03 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S2_STA__1S ASCENDING T003_F288 M02 (7 scene(s), cycles=['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'])
  S2_STA__1S ASCENDING T003_F288 

## Inspect STAC item properties



In [20]:
# Print every property of the first available master item so you can
# identify the correct keys for baseline and incidence angle.
if pairs:
    first_master = next(iter(pairs.values()))[0]
    props = dict(first_master.properties)
    print(f"Item: {first_master.id}")
    print(f"\nAll properties ({len(props)} keys):")
    for k, v in sorted(props.items()):
        print(f"  {k!r:55s}  {v!r}")
    print(f"\nAll asset keys:")
    for k, v in sorted(first_master.assets.items()):
        print(f"  {k!r:40s}  href={v.href[:80]!r}")


Item: BIO_S1_STA__1S_20260219T045815_20260219T045836_T_G01_M02_C04_T003_F290_02_DUC3F2

All properties (37 keys):
  'auth:schemes'                                           {'s3': {'type': 's3'}, 'oidc': {'openIdConnectUrl': 'https://iam.ascend.icsgate.eu/realms/esa-maap/.well-known/openid-configuration', 'type': 'openIdConnect'}}
  'constellation'                                          'Biomass'
  'created'                                                '2026-07-10T15:37:50Z'
  'datetime'                                               '2026-02-19T04:58:15.713Z'
  'end_datetime'                                           '2026-02-19T04:58:36.118Z'
  'eofeos:global_coverage_id'                              '1'
  'eofeos:is_coregistration_primary'                       True
  'eofeos:major_cycle_id'                                  '2'
  'eofeos:mission_phase'                                   'TOMOGRAPHIC'
  'eofeos:orbit_drift_flag'                                False
  'eofeos:repeat

## Streaming and physics helpers

### Asset discovery and band streaming


In [22]:
POL_INDEX = {"HH": 0, "HV": 1, "VH": 2, "VV": 3}


def _find_asset_href(item, preferred_keys, contains_any):
    for key in preferred_keys:
        if key in item.assets:
            return item.assets[key].href
    for key in item.assets:
        if any(tok in key.lower() for tok in contains_any):
            return item.assets[key].href
    raise KeyError(f"No asset matched for {item.id}. Keys: {list(item.assets)}")


def find_abs_asset_href(item):
    return _find_asset_href(item,
        ["i_abs", "I_abs", "abs", "amplitude", "measurement_i_abs", "enclosure_13"],
        ["abs"])


def find_phase_asset_href(item):
    return _find_asset_href(item,
        ["i_pha", "I_pha", "phase", "pha", "measurement_i_pha", "enclosure_14"],
        ["pha", "phase"])


def stream_bands(href, token, decimate=1):
    """Stream all bands of a remote GeoTIFF via GDAL HTTP.

    If decimate > 1, read a subsampled grid (every `decimate`-th pixel, nearest
    resampling) instead of the full array. Nearest keeps amplitude and phase
    co-located so the complex reconstruction stays valid. This cuts memory, and
    also download volume when the source carries overviews.
    """
    from rasterio.enums import Resampling
    with rio.Env(
        GDAL_HTTP_HEADERS=f"Authorization: Bearer {token}",
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_NUM_THREADS="1",
    ):
        if decimate and int(decimate) > 1:
            d = int(decimate)
            with rio.open(href) as ds:
                oh = max(1, ds.height // d)
                ow = max(1, ds.width // d)
                arr = ds.read(out_shape=(ds.count, oh, ow),
                              resampling=Resampling.nearest).astype(np.float32)
        else:
            da = riox.open_rasterio(href, chunks=None, masked=False).load()
            arr = da.values.astype(np.float32)   # (n_bands, ny, nx)
            del da
    gc.collect()
    return arr

In [23]:
def unpack_pair_key(key):
    """
    Supports:

    Old:
        (pt, track, frame)

    Older:
        (pt, orbit, track, frame, dup_idx)

    New:
        (pt, orbit, track, frame, major_cycle, dup_idx)
    """

    if len(key) == 3:
        pt, track, frame = key
        orbit = None
        major_cycle = None
        dup_idx = None

    elif len(key) == 5:
        pt, orbit, track, frame, dup_idx = key
        major_cycle = None

    elif len(key) == 6:
        pt, orbit, track, frame, major_cycle, dup_idx = key

    else:
        raise ValueError(
            f"Unexpected pair key format: {key}. "
            "Expected (pt, track, frame), "
            "(pt, orbit, track, frame, dup_idx) or "
            "(pt, orbit, track, frame, major_cycle, dup_idx)."
        )

    return pt, orbit, track, frame, major_cycle, dup_idx


### ICAL phase correction and coherence



In [24]:
def cpx_coherence(e1, e2, window_px=5):
    """
    Windowed complex interferometric coherence.
    Equivalent to bioqlk.cpx_coherence(e1, e2).
    """
    cross = e1 * np.conj(e2)
    num = (uniform_filter(cross.real, window_px)
           + 1j * uniform_filter(cross.imag, window_px))
    denom = np.sqrt(
        np.maximum(uniform_filter(np.abs(e1) ** 2, window_px), 0)
        * np.maximum(uniform_filter(np.abs(e2) ** 2, window_px), 0)
    )
    return np.where(denom > 1e-10, num / denom, 0j)

In [25]:

# Tiled full-resolution Method 2 (I2EM) retrieval


def run_i2em_inversion_tiled(sigma0_hh_db, sigma0_vv_db, gamma_mag, theta_deg,
                             i2em_lut, b_perp_m, r_slant_m,
                             tile_rows=I2EM_TILE_ROWS, i2em_kwargs=None):
    """Joint HH+VV I2EM retrieval over horizontal row tiles of the FULL-resolution
    scene, so the grid-search working set never materialises for the whole scene
    at once. Output arrays are full resolution — tiling only bounds peak memory.
    Halves the tile size and retries on MemoryError. Each pixel is independent,
    and theta-binning happens within each tile (tiles keep the full column range),
    so tiling does not change the result.

    Returns eps_est, ks_est, depth_m (each ny x nx).
    """
    if i2em_kwargs is None:
        i2em_kwargs = I2EM_KWARGS
    ny, nx = sigma0_vv_db.shape
    n_tiles_est = int(np.ceil(ny / max(tile_rows, 1)))
    print(f"    Method 2 (I2EM) tiled retrieval at full resolution: "
          f"{ny} x {nx} px, ~{tile_rows} rows/tile ({n_tiles_est} tile(s) est.)")

    eps_out = np.full((ny, nx), np.nan, dtype=np.float64)
    ks_out = np.full((ny, nx), np.nan, dtype=np.float64)
    depth_out = np.full((ny, nx), np.nan, dtype=np.float64)

    row = 0
    while row < ny:
        r0, r1 = row, min(row + tile_rows, ny)
        while True:
            try:
                res = retrieve_eps_ks_mv_and_depth_from_i2em_lut(
                    sigma0_hh=sigma0_hh_db[r0:r1],
                    sigma0_vv=sigma0_vv_db[r0:r1],
                    gamma_mag=gamma_mag[r0:r1],
                    theta_deg=theta_deg[r0:r1],
                    i2em_lut=i2em_lut,
                    b_perp_m=b_perp_m,
                    r_slant_m=r_slant_m,
                    **i2em_kwargs,
                )
                break
            except MemoryError:
                if (r1 - r0) <= MIN_TILE_ROWS:
                    raise
                r1 = r0 + max(MIN_TILE_ROWS, (r1 - r0) // 2)
                print(f"    MemoryError on tile [{r0}:{r1}] - halving tile size and retrying")
                gc.collect()

        eps_out[r0:r1] = np.asarray(res["eps_est"], dtype=float)
        ks_out[r0:r1] = np.asarray(res["ks_est"], dtype=float)
        depth_out[r0:r1] = np.asarray(res["depth_m"], dtype=float)
        del res
        gc.collect()

        # if this tile had to shrink, keep the smaller size for subsequent tiles too
        tile_rows = min(tile_rows, r1 - r0) if (r1 - r0) < tile_rows else tile_rows
        row = r1

    print(f"    Method 2 retrieval complete: {ny} x {nx} px, full resolution retained")
    return eps_out, ks_out, depth_out



def mv_from_eps_lut_tiled(eps_est, dielectric_model, tile_rows=None, **dielectric_kwargs):
    """Row-tiled wrapper around mv_from_eps_lut.

    mv_from_eps_lut does a nearest-grid lookup by broadcasting an
    (n_mv, ny, nx) array (n_mv ~ 351). At full BIOMASS resolution that single
    temporary is tens of GiB (the MemoryError source). The lookup is purely
    per-pixel, so slicing the eps' map into row tiles gives an identical result
    while capping the temporary to (n_mv, tile_rows, nx). Halves on MemoryError.
    """
    if tile_rows is None:
        tile_rows = MV_TILE_ROWS
    eps_est = np.asarray(eps_est, dtype=float)
    ny, nx = eps_est.shape
    mv_out = np.full((ny, nx), np.nan, dtype=float)
    print(f"    eps'->mv ({dielectric_model}) tiled lookup: {ny} x {nx} px, ~{tile_rows} rows/tile")
    row = 0
    while row < ny:
        r0, r1 = row, min(row + tile_rows, ny)
        while True:
            try:
                mv_out[r0:r1] = mv_from_eps_lut(
                    eps_est[r0:r1], dielectric_model=dielectric_model, **dielectric_kwargs)
                break
            except MemoryError:
                if (r1 - r0) <= MIN_TILE_ROWS:
                    raise
                r1 = r0 + max(MIN_TILE_ROWS, (r1 - r0) // 2)
                print(f"    MemoryError in mv lookup on tile [{r0}:{r1}] - halving and retrying")
                gc.collect()
        tile_rows = min(tile_rows, r1 - r0) if (r1 - r0) < tile_rows else tile_rows
        row = r1
    return mv_out


## Core depth inversion — Method 2 (I2EM physical LUT), streaming version




In [26]:
def _scene_averages(eps, mv, depth):
    """nan-aware mean & median of the three retrieved fields for one scene."""
    def stat(a):
        a = np.asarray(a, float)
        finite = np.isfinite(a)
        if not finite.any():
            return (np.nan, np.nan, 0)
        return (float(np.nanmean(a)), float(np.nanmedian(a)), int(finite.sum()))
    e_mean, e_med, e_n = stat(eps)
    m_mean, m_med, _ = stat(mv)
    d_mean, d_med, _ = stat(depth)
    return {
        "eps_mean": e_mean, "eps_median": e_med,
        "mv_mean": m_mean, "mv_median": m_med,
        "depth_mean_m": d_mean, "depth_median_m": d_med,
        "n_valid_px": e_n,
    }


def run_depth_inversion_from_stac(
    master_item,
    secondary_item,
    token,
    decimate=1,
    b_perp_override=None,
    b_perp_source=None,
):
    """Method 2 (I2EM LUT) depth inversion for one matched (master, secondary) STAC pair.

    Important:
        If b_perp_override is provided, that value is used operationally.
        This should be the normalBaseline matched to the same primaryImage and
        secondaryImage used here.
    """

    import time

    _t0 = time.perf_counter()
    _stage = {}

    def _lap(label, t):
        now = time.perf_counter()
        dt = now - t
        _stage[label] = dt
        print(f"    [time] {label:<22} {dt:8.1f} s")
        return now

    print(f"    Streaming master:    {master_item.id}")
    print(f"    Streaming secondary: {secondary_item.id}")

    if decimate and int(decimate) > 1:
        print(f"    DECIMATE={int(decimate)} -> reading subsampled grid (fast/preview mode)")

    # eam amplitude and phase for both scenes 
    _t = time.perf_counter()

    abs_m = stream_bands(find_abs_asset_href(master_item), token, decimate)
    pha_m = stream_bands(find_phase_asset_href(master_item), token, decimate)

    abs_s = stream_bands(find_abs_asset_href(secondary_item), token, decimate)
    pha_s = stream_bands(find_phase_asset_href(secondary_item), token, decimate)

    _t = _lap("stream (4 files)", _t)

    # onstruct complex SLC for VV only 
    vv_m = (
        abs_m[POL_INDEX["VV"]].astype(np.complex64)
        * np.exp(1j * pha_m[POL_INDEX["VV"]])
    )

    vv_s = (
        abs_s[POL_INDEX["VV"]].astype(np.complex64)
        * np.exp(1j * pha_s[POL_INDEX["VV"]])
    )

    # interferometric coherence 
    ny_m, nx_m = vv_m.shape
    ny_s, nx_s = vv_s.shape

    rows, cols = min(ny_m, ny_s), min(nx_m, nx_s)

    if (ny_m, nx_m) != (ny_s, nx_s):
        print(
            f"    NOTE: master {(ny_m, nx_m)} vs secondary {(ny_s, nx_s)} differ - "
            f"cropping to ({rows}, {cols}); if this exceeds a few px the pair may not be grid-aligned"
        )
    else:
        print(f"    master & secondary identically sized {(ny_m, nx_m)} - crop is a no-op")

    coh = cpx_coherence(
        vv_m[:rows, :cols],
        vv_s[:rows, :cols],
    )

    gamma_mag = np.abs(coh).astype(np.float32)

    del vv_m, vv_s, coh
    gc.collect()

    _t = _lap("coherence (VV only)", _t)

    # & VV sigma0 from master amplitude 
    hh_amp = abs_m[POL_INDEX["HH"]][:rows, :cols]
    vv_amp = abs_m[POL_INDEX["VV"]][:rows, :cols]

    sigma0_hh_native = (hh_amp ** 2).astype(np.float32)
    sigma0_vv_native = (vv_amp ** 2).astype(np.float32)

    if USE_LEE_FILTER:
        sigma0_hh_native = lee_filter(sigma0_hh_native, size=LEE_SIZE)
        sigma0_vv_native = lee_filter(sigma0_vv_native, size=LEE_SIZE)

    # oherent multilook before inversion 
    if (ML_AZ, ML_RG) != (1, 1):
        _pre = (rows, cols)

        sigma0_hh_native = _multilook(
            sigma0_hh_native,
            ML_AZ,
            ML_RG,
            valid_max=10.0,
        )

        sigma0_vv_native = _multilook(
            sigma0_vv_native,
            ML_AZ,
            ML_RG,
            valid_max=10.0,
        )

        gamma_mag = _multilook(
            gamma_mag,
            ML_AZ,
            ML_RG,
        ).astype(np.float32)

        print(
            f"    multilook {ML_AZ}x{ML_RG} ({ML_AZ * ML_RG} looks): "
            f"{_pre[0]}x{_pre[1]} -> "
            f"{sigma0_vv_native.shape[0]}x{sigma0_vv_native.shape[1]} px"
        )

    # VV backscatter layer for KMZ / preview
    vv_db_full = 10.0 * np.log10(
        np.clip(sigma0_vv_native, 1e-10, None)
    )

    del abs_m, abs_s, pha_m, pha_s, hh_amp, vv_amp
    gc.collect()

    _t = _lap("sigma0 + Lee + multilook", _t)

    target_shape = sigma0_vv_native.shape

    if gamma_mag.shape != target_shape:
        zf = (
            target_shape[0] / gamma_mag.shape[0],
            target_shape[1] / gamma_mag.shape[1],
        )
        gamma_mag = zoom(gamma_mag, zf, order=1)



    geom = get_baseline_and_slant(master_item, token)

    if isinstance(geom, dict):
        b_perp_extracted = geom.get("b_perp_m", None)
        r_slant_m = geom.get("r_slant_m", None)
        r_slant_source = geom.get("r_slant_source", "master_geometry")
    else:
        b_perp_extracted, r_slant_m = geom
        r_slant_source = "master_LUT_or_geometry"

    if b_perp_override is not None:
        b_perp_m = float(b_perp_override)
        b_perp_used_source = b_perp_source or "annotation_matched_normalBaseline"

        print(
            f"    Using annotation-matched B_perp: "
            f"{b_perp_m:+.2f} m [{b_perp_used_source}]"
        )

        if b_perp_extracted is not None and np.isfinite(b_perp_extracted):
            print(
                f"    Baseline re-extracted from selected master only: "
                f"{float(b_perp_extracted):+.2f} m "
                f"(comparison only; not used)"
            )

    else:
        b_perp_m = float(b_perp_extracted)
        b_perp_used_source = "get_baseline_and_slant_master_only"

        print(
            f"    WARNING: using baseline extracted from master only: "
            f"{b_perp_m:+.2f} m"
        )

    if not np.isfinite(b_perp_m) or abs(b_perp_m) < 1e-3:
        raise RuntimeError(
            f"Invalid B_perp used for inversion: {b_perp_m}. "
            "Check baseline matching and avoid self-pairs or zero-baseline pairs."
        )

    if r_slant_m is None or not np.isfinite(r_slant_m):
        raise RuntimeError(
            "No valid slant range available for vertical-wavenumber calculation."
        )

    inc_arr = get_incidence_angle_from_nc(
        master_item,
        token,
        target_shape,
    )

    # ma0 -> dB, masking fill / unphysical values 
    sigma0_hh_db, _ = clean_sigma0_linear_to_db(sigma0_hh_native)
    sigma0_vv_db, _ = clean_sigma0_linear_to_db(sigma0_vv_native)

    del sigma0_hh_native, sigma0_vv_native
    gc.collect()

    _t = _lap("geometry + dB prep", _t)

    # hod 2: joint HH+VV I2EM retrieval 
    eps_est, ks_est, depth_m = run_i2em_inversion_tiled(
        sigma0_hh_db,
        sigma0_vv_db,
        gamma_mag,
        inc_arr,
        i2em_lut,
        b_perp_m,
        r_slant_m,
    )

    del sigma0_hh_db, sigma0_vv_db
    gc.collect()

    _t = _lap("I2EM retrieval", _t)

    # umetric moisture from retrieved eps' 
    mv_kwargs = (
        dict(sand_pct=85.0, clay_pct=5.0, pb=1.3)
        if DIELECTRIC_MODEL == "dobson"
        else {}
    )

    mv_est = mv_from_eps_lut_tiled(
        eps_est,
        dielectric_model=DIELECTRIC_MODEL,
        **mv_kwargs,
    )

    _t = _lap("mv lookup", _t)

    # -scene averages 
    averages = _scene_averages(
        eps_est,
        mv_est,
        depth_m,
    )

    print(
        f"    scene avg  eps'={averages['eps_mean']:.3f}  "
        f"mv={averages['mv_mean']:.4f}  "
        f"depth={averages['depth_mean_m']:.3f} m   "
        f"(median: eps'={averages['eps_median']:.3f}, "
        f"mv={averages['mv_median']:.4f}, "
        f"depth={averages['depth_median_m']:.3f} m; "
        f"n_valid={averages['n_valid_px']})"
    )

    total = time.perf_counter() - _t0

    breakdown = "  ".join(
        f"{k.split()[0]} {v:.0f}s"
        for k, v in _stage.items()
    )

    print(
        f"    [time] TOTAL              {total:8.1f} s  "
        f"({total / 60:.1f} min)   [{breakdown}]"
    )

    return {
        "vv_db": np.asarray(vv_db_full, dtype=np.float32),
        "mv": np.asarray(mv_est, dtype=np.float32),
        "eps": np.asarray(eps_est, dtype=np.float32),
        "depth": np.asarray(depth_m, dtype=np.float32),
        "ks": np.asarray(ks_est, dtype=np.float32),
        "b_perp_m": b_perp_m,
        "b_perp_source": b_perp_used_source,
        "r_slant_m": r_slant_m,
        "r_slant_source": r_slant_source,
        "averages": averages,
    }


## Export helpers


In [28]:
def _downsample_for_png(array, max_dim=None):
    """Stride a big map down before rendering.

    The overlay is saved at ~2400 px wide no matter what, so handing matplotlib a
    20756 x 1367 array just burns RAM and time building an RGBA buffer it then
    throws away. Striding first is visually identical in Google Earth.
    """
    if max_dim is None:
        max_dim = PNG_MAX_DIM
    step = max(1, int(np.ceil(max(array.shape) / float(max_dim))))
    return array[::step, ::step] if step > 1 else array


def robust_limits(array, low=2, high=98, max_px=None):
    """2/98 percentiles from a strided subsample.

    np.nanpercentile sorts a full copy of the array - ~227 MB per call at full
    resolution, four times per pair. A couple of million pixels give the same
    display limits for a fraction of the memory.
    """
    if max_px is None:
        max_px = PCTL_SUBSAMPLE
    a = np.asarray(array)
    if a.size > max_px:
        step = int(np.ceil(np.sqrt(a.size / float(max_px))))
        a = a[::step, ::step]
    a = a[np.isfinite(a)]
    if a.size == 0:
        return (0.0, 1.0)
    return tuple(np.percentile(a, [low, high]))


def save_overlay_png(array, png_path, cmap, vmin, vmax):
    array = _downsample_for_png(array)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(array, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_axis_off()
    fig.savefig(png_path, dpi=200, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close(fig)
    del array
    gc.collect()


def build_multilayer_kmz(layers, coords_str, kmz_path, doc_name):
    """layers: list of (name, png_path, visible) tuples sharing one footprint quad."""
    overlay_blocks = []
    for name, png_path, visible in layers:
        vis_flag = 1 if visible else 0
        overlay_blocks.append(f"""    <GroundOverlay>
      <name>{name}</name>
      <visibility>{vis_flag}</visibility>
      <Icon><href>{png_path.name}</href></Icon>
      <gx:LatLonQuad>
        <coordinates>{coords_str}</coordinates>
      </gx:LatLonQuad>
    </GroundOverlay>""")

    overlays_joined = "\n".join(overlay_blocks)
    kml_text = f"""<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2" xmlns:gx="http://www.google.com/kml/ext/2.2">
  <Document>
    <name>{doc_name}</name>
    <open>1</open>
{overlays_joined}
  </Document>
</kml>
"""
    with zipfile.ZipFile(kmz_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("doc.kml", kml_text)
        for _, png_path, _ in layers:
            archive.write(png_path, arcname=png_path.name)



def valid_bbox(array, min_frac=0.0):
    """Bounding box (r0, r1, c0, c1) of rows/columns containing valid data.

    A row/column is kept if its fraction of finite pixels exceeds `min_frac`.
    min_frac=0.0 keeps anything with at least one finite pixel; raise it to
    ~0.05 to also drop nearly-empty edge lines. Slices are half-open (r0:r1).
    """
    finite = np.isfinite(np.asarray(array))
    if not finite.any():
        return (0, array.shape[0], 0, array.shape[1])
    row_ok = finite.mean(axis=1) > min_frac
    col_ok = finite.mean(axis=0) > min_frac
    rows = np.flatnonzero(row_ok)
    cols = np.flatnonzero(col_ok)
    if rows.size == 0 or cols.size == 0:
        return (0, array.shape[0], 0, array.shape[1])
    return (int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1)


def crop_maps_to_valid(maps, ref_key="eps", min_frac=0.0, verbose=True):
    """Crop every 2-D layer in `maps` to one shared valid-data bbox.

    The bbox is taken from `ref_key` (a retrieval output, so its NaNs mark the
    fill margins) and applied identically to all layers, keeping them aligned.
    Returns a new dict; the original `maps` is not modified.
    """
    ref = maps[ref_key]
    r0, r1, c0, c1 = valid_bbox(ref, min_frac=min_frac)
    out = {}
    for k, v in maps.items():
        if isinstance(v, np.ndarray) and v.ndim == 2 and v.shape == ref.shape:
            out[k] = v[r0:r1, c0:c1]
        else:
            out[k] = v
    if verbose:
        dr, dc = ref.shape[0] - (r1 - r0), ref.shape[1] - (c1 - c0)
        print(f"display crop: {ref.shape} -> {(r1 - r0, c1 - c0)}  "
              f"(trimmed {dr} row(s), {dc} column(s); rows {r0}:{r1}, cols {c0}:{c1})")
    out["_crop_bbox"] = (r0, r1, c0, c1)
    return out


In [32]:
from pathlib import Path
import tempfile
import gc
import numpy as np


# 
# Output directory
# 

OUTPUT_DIR = Path(
    r"C:\Users\Femke.Graveland\OneDrive - ESA\Documents\DATA\1C Nambia\Google Earth Output (MAAP DI Method2_inc_Nam_perp_baseline_corrected_2500_scaled_large area)"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 
# Pair-key helper
# 

def unpack_pair_key(key):
    """
    Supports multiple pair key formats.

    Old:
        (pt, track, frame)

    Older corrected:
        (pt, orbit, track, frame, dup_idx)

    New with major cycle:
        (pt, orbit, track, frame, major_cycle, dup_idx)

    Returns
    
    pt, orbit, track, frame, major_cycle, dup_idx
    """

    if len(key) == 3:
        pt, track, frame = key
        orbit = None
        major_cycle = None
        dup_idx = None

    elif len(key) == 5:
        pt, orbit, track, frame, dup_idx = key
        major_cycle = None

    elif len(key) == 6:
        pt, orbit, track, frame, major_cycle, dup_idx = key

    else:
        raise ValueError(
            f"Unexpected pair key format: {key}. "
            "Expected either "
            "(pt, track, frame), "
            "(pt, orbit, track, frame, dup_idx), or "
            "(pt, orbit, track, frame, major_cycle, dup_idx)."
        )

    return pt, orbit, track, frame, major_cycle, dup_idx


def pair_key_label(key):
    """
    Human-readable label for pair key.
    """
    pt, orbit, t, f, mc, dup_idx = unpack_pair_key(key)

    orbit_txt = orbit if orbit is not None else "UNKNOWN_ORBIT"
    mc_txt = mc if mc is not None else "M??"
    dup_txt = dup_idx if dup_idx is not None else "D??"

    return f"{pt} {orbit_txt} T{t}_F{f} {mc_txt} {dup_txt}"


def base_name_for_key(key):
    """
    Build base output name from a key, without extension or layer suffix.
    """
    pt, orbit, t, f, mc, dup_idx = unpack_pair_key(key)

    orbit_short = {
        "ASCENDING": "ASC",
        "DESCENDING": "DES",
        "UNKNOWN": "UNK",
        None: "UNK",
    }.get(orbit, orbit)

    mc_txt = mc if mc is not None else "MXX"
    dup_txt = dup_idx if dup_idx is not None else "DXX"

    return f"{pt}_{orbit_short}_T{t}_F{f}_{mc_txt}_{dup_txt}"


def kmz_path_for_key_and_layer(key, layer_short):
    """
    Build output KMZ path for an individual variable layer.
    """
    return OUTPUT_DIR / f"{base_name_for_key(key)}_{layer_short}.kmz"


# 
# Require baseline_by_key and pairs from corrected scene-selection cell
# 

if "baseline_by_key" not in globals():
    raise RuntimeError(
        "baseline_by_key is not defined. Run the baseline-matched scene-selection "
        "cell before running this batch export cell."
    )

if "pairs" not in globals() or not pairs:
    raise RuntimeError(
        "pairs is not defined or empty. Run the baseline-matched scene-selection "
        "cell before running this batch export cell."
    )


# 
# Remove invalid / self-pair / near-zero baseline pairs before queueing
# 

valid_pairs = {}
skipped_invalid_baseline = {}

for key, scenes in pairs.items():

    if key not in baseline_by_key:
        skipped_invalid_baseline[key] = "missing baseline_by_key entry"
        continue

    if scenes is None or len(scenes) < 2:
        skipped_invalid_baseline[key] = "less than two scenes"
        continue

    master_item, secondary_item = scenes[0], scenes[1]
    binfo = baseline_by_key[key]
    b_perp = binfo.get("b_perp_m", None)

    if master_item.id == secondary_item.id:
        skipped_invalid_baseline[key] = "self-pair master == secondary"
        continue

    if b_perp is None or not np.isfinite(b_perp) or abs(float(b_perp)) < 1e-3:
        skipped_invalid_baseline[key] = f"invalid or near-zero B_perp: {b_perp}"
        continue

    valid_pairs[key] = scenes


if skipped_invalid_baseline:
    print("Skipped invalid baseline-matched pair(s):")
    for key, reason in skipped_invalid_baseline.items():
        print(f"  {pair_key_label(key)}: {reason}")
    print()


pairs_for_run = valid_pairs




try:
    _summary_by_key
except NameError:
    _summary_by_key = {}

try:
    exported
except NameError:
    exported = []


token = get_token(CREDENTIALS_FILE)



EPS_VMIN, EPS_VMAX = 2.0, 10.0
DEPTH_VMIN, DEPTH_VMAX = 0.0, 15.0
MV_VMIN, MV_VMAX = 0.0, 0.15



LAYER_SPECS = [
    {
        "label": "Dielectric constant eps'",
        "short": "eps",
        "map_key": "eps",
        "cmap": "viridis",
        "vmin": EPS_VMIN,
        "vmax": EPS_VMAX,
    },
    {
        "label": "Effective penetration depth",
        "short": "depth",
        "map_key": "depth",
        "cmap": "plasma",
        "vmin": DEPTH_VMIN,
        "vmax": DEPTH_VMAX,
    },
    {
        "label": "Volumetric moisture mv",
        "short": "mv",
        "map_key": "mv",
        "cmap": "cividis",
        "vmin": MV_VMIN,
        "vmax": MV_VMAX,
    },
]




_all_pairs = list(pairs_for_run.items())

if SKIP_EXISTING:
    _pending = []

    for k, v in _all_pairs:
        expected_files = [
            kmz_path_for_key_and_layer(k, spec["short"])
            for spec in LAYER_SPECS
        ]

        # Only skip the pair if eps, depth, and mv files all already exist.
        if not all(path.exists() for path in expected_files):
            _pending.append((k, v))

else:
    _pending = _all_pairs


_n_done = len(_all_pairs) - len(_pending)
_queue = _pending[:MAX_PAIRS] if MAX_PAIRS else _pending

print(
    f"pairs total: {len(_all_pairs)} | already exported: {_n_done} | "
    f"remaining: {len(_pending)} | this run: {len(_queue)}  "
    f"(BATCH_DECIMATE={BATCH_DECIMATE}, SKIP_EXISTING={SKIP_EXISTING})"
)

if not _queue:
    print(
        "Nothing to do. Every valid baseline-matched pair already has separate "
        "eps, depth, and mv KMZ files. Set SKIP_EXISTING=False to force reprocessing."
    )

print()




for i_pair, (key, scenes) in enumerate(_queue):

    pt, orbit, t, f, mc, dup_idx = unpack_pair_key(key)

    if i_pair > 0 and i_pair % TOKEN_REFRESH_EVERY == 0:
        print("\nRefreshing access token...")
        token = get_token(CREDENTIALS_FILE)

    master_item, secondary_item = scenes[0], scenes[1]

    binfo = baseline_by_key[key]
    b_perp_override = float(binfo["b_perp_m"])
    b_perp_source = binfo["b_perp_source"]

    print(f"\n{'=' * 60}")
    print(
        f"[{i_pair + 1}/{len(_queue)}] "
        f"{pt} {orbit} T{t}_F{f} {mc} {dup_idx} | "
        f"{cycle_label(master_item)} -> {cycle_label(secondary_item)}"
    )
    print(
        f"    Using baseline-matched B_perp: "
        f"{b_perp_override:+.2f} m [{b_perp_source}]"
    )
    print(f"{'=' * 60}")

    maps = None

    try:
        coords_str = get_item_quad_coordinates(master_item)

        maps = run_depth_inversion_from_stac(
            master_item,
            secondary_item,
            token,
            decimate=BATCH_DECIMATE,
            b_perp_override=b_perp_override,
            b_perp_source=b_perp_source,
        )

        print("  available map keys:", list(maps.keys()))

        # 
        # Collect per-scene averages, keyed so reprocessing replaces its row
        # 

        avg = maps["averages"]

        _summary_by_key[key] = {
            "product_type": pt,
            "orbit": orbit,
            "track": t,
            "frame": f,
            "major_cycle": mc,
            "dup_idx": dup_idx,
            "master": cycle_label(master_item),
            "secondary": cycle_label(secondary_item),
            "master_id": master_item.id,
            "secondary_id": secondary_item.id,
            **avg,
            "b_perp_m": maps["b_perp_m"],
            "b_perp_source": maps.get("b_perp_source", "unknown"),
            "r_slant_m": maps["r_slant_m"],
            "r_slant_source": maps.get("r_slant_source", "unknown"),
        }

        # 
        # Create one separate KMZ for eps, depth, and mv
        # 

        for spec in LAYER_SPECS:

            label = spec["label"]
            short = spec["short"]
            map_key = spec["map_key"]
            cmap = spec["cmap"]
            vmin = spec["vmin"]
            vmax = spec["vmax"]

            if map_key not in maps:
                print(
                    f"  WARNING: map key '{map_key}' not found. "
                    f"Skipping separate {short} export."
                )
                continue

            arr = maps[map_key]

            if vmin is None or vmax is None:
                v0, v1 = robust_limits(arr)
                scale_type = "scene robust"
            else:
                v0, v1 = vmin, vmax
                scale_type = "fixed global"

            kmz_path = kmz_path_for_key_and_layer(key, short)

            if SKIP_EXISTING and kmz_path.exists():
                print(f"  already exists, skipping: {kmz_path.name}")
                continue

            print(
                f"  exporting: {label:35s} | "
                f"map key: {map_key:8s} | "
                f"scale: {scale_type:13s} | "
                f"vmin={v0:.3f}, vmax={v1:.3f}"
            )

            with tempfile.TemporaryDirectory() as tmp_dir:
                tmp_dir = Path(tmp_dir)
                png_path = tmp_dir / f"{short}.png"

                save_overlay_png(
                    arr,
                    png_path,
                    cmap=cmap,
                    vmin=v0,
                    vmax=v1,
                )

                layer_files = [
                    (label, png_path, True)
                ]

                doc_name = (
                    f"DI(M2) {short.upper()} {pt} {orbit} T{t} F{f} {mc} {dup_idx} "
                    f"({cycle_label(master_item)} -> {cycle_label(secondary_item)})"
                )

                build_multilayer_kmz(
                    layer_files,
                    coords_str,
                    kmz_path,
                    doc_name=doc_name,
                )

            print(f"  saved: {kmz_path}")

            exported.append(
                {
                    "key": key,
                    "product_type": pt,
                    "orbit": orbit,
                    "track": t,
                    "frame": f,
                    "major_cycle": mc,
                    "dup_idx": dup_idx,
                    "layer": short,
                    "kmz_path": kmz_path,
                }
            )

    except Exception as e:
        print(f"  ERROR: {e}")

    finally:
        # Free maps even when export raises, so one bad pair cannot leave
        # large arrays resident in memory.
        maps = None
        gc.collect()




scene_summaries = list(_summary_by_key.values())

_remaining = len(_pending) - len(_queue)

print(f"\n{'=' * 60}")
print(
    f"Done this run: {len(_queue)} pair(s) attempted, "
    f"{len(scene_summaries)} summarised total."
)
print(f"Output: {OUTPUT_DIR.resolve()}")

if _remaining > 0:
    print(
        f"{_remaining} pair(s) still pending. Re-run this cell to continue "
        f"(finished pairs are skipped automatically)."
    )

pairs total: 4 | already exported: 0 | remaining: 4 | this run: 1  (BATCH_DECIMATE=1, SKIP_EXISTING=True)


[1/1] S1_STA__1S ASCENDING T003_F290 M02 D01 | C04 -> C01
    Using baseline-matched B_perp: -2424.32 m [normalBaseline_preferred_C04_to_C03_or_C05_intermediate_baseline]
    Streaming master:    BIO_S1_STA__1S_20260219T045815_20260219T045836_T_G01_M02_C04_T003_F290_02_DUC3F2
    Streaming secondary: BIO_S1_STA__1S_20260210T045809_20260210T045830_T_G01_M02_C01_T003_F290_02_DUC38Q
    [time] stream (4 files)          210.9 s
    master & secondary identically sized (20789, 1379) - crop is a no-op


C:\Users\Femke.Graveland\AppData\Local\Temp\ipykernel_28420\2330520613.py:13: RuntimeWarning: divide by zero encountered in divide
  return np.where(denom > 1e-10, num / denom, 0j)


    [time] coherence (VV only)         7.0 s
    multilook 12x2 (24 looks): 20789x1379 -> 1732x689 px
    [time] sigma0 + Lee + multilook      9.5 s
    Baseline from annotation XML: 0.00 m
    Slant range from LUT NetCDF: 741580.74 m
    Using annotation-matched B_perp: -2424.32 m [normalBaseline_preferred_C04_to_C03_or_C05_intermediate_baseline]
    Baseline re-extracted from selected master only: +0.00 m (comparison only; not used)
    Incidence angle from LUT NetCDF geometry group
    [time] geometry + dB prep         27.3 s
    Method 2 (I2EM) tiled retrieval at full resolution: 1732 x 689 px, ~512 rows/tile (4 tile(s) est.)
    Method 2 retrieval complete: 1732 x 689 px, full resolution retained
    [time] I2EM retrieval            264.4 s
    eps'->mv (topp) tiled lookup: 1732 x 689 px, ~256 rows/tile
    [time] mv lookup                   6.0 s
    scene avg  eps'=6.030  mv=0.0935  depth=4.711 m   (median: eps'=5.281, mv=0.0830, depth=4.285 m; n_valid=1109612)
    [time] TOTAL 